In [13]:
import pandas as pd
import numpy as np

def generate_data_quality_report(filepath):
    df = pd.read_csv(filepath)
    
    print("=" * 60)
    print("DATA QUALITY REPORT")
    print("=" * 60)
    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
    
    # Missing Values & Types
    summary_df = pd.DataFrame({
        'Data Type': df.dtypes,
        'Missing Count': df.isna().sum(),
        'Missing %': (df.isna().mean() * 100).round(2),
        'Unique Count': df.nunique()
    })
    print("--- COLUMN ANALYSIS ---")
    print(summary_df)
    print(f"\nDuplicate Rows: {df.duplicated().sum()}")
    
    print("\n--- NUMERIC SUMMARY ---")
    print(df.describe().T[['mean', 'std', 'min', '50%', 'max']])
    
    print("\n--- CATEGORICAL SUMMARY ---")
    cat_cols = df.select_dtypes(include=['object', 'string']).columns
    if len(cat_cols) > 0:
        print(df[cat_cols].describe().T)
    else:
        print("No categorical columns detected.")

# Execute the function call:
generate_data_quality_report("student_performance.csv")

DATA QUALITY REPORT
Shape: 5 rows, 7 columns

--- COLUMN ANALYSIS ---
                Data Type  Missing Count  Missing %  Unique Count
student_id            str              0        0.0             5
study_hours       float64              1       20.0             4
attendance          int64              0        0.0             5
previous_score      int64              0        0.0             5
city                  str              0        0.0             4
internet_access       str              0        0.0             2
final_score         int64              0        0.0             5

Duplicate Rows: 0

--- NUMERIC SUMMARY ---
                  mean        std   min    50%   max
study_hours      5.125   2.528998   2.0   5.25   8.0
attendance      82.800  13.953494  60.0  87.00  95.0
previous_score  71.000  22.405357  40.0  72.00  98.0
final_score     73.800  21.463923  45.0  81.00  95.0

--- CATEGORICAL SUMMARY ---
                count unique     top freq
student_id          5 

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Create/Load Sample Dataset
data = {
    'student_id': ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S001'],
    'study_hours': [4.5, 6.0, np.nan, 2.0, 45.0, 3.5, 4.5], # 45.0 is an outlier
    'attendance': [87, 92, -15, 60, 95, np.nan, 87],        # -15 is an invalid value
    'previous_score': [72, 85, 60, 40, 98, 55, 72],
    'city': ['Lahore', ' lahore ', 'Karachi', 'ISLAMABAD', 'Lahore', 'karachi', 'Lahore'],
    'internet_access': ['Yes', 'Yes', 'No', 'Yes', np.nan, 'No', 'Yes'],
    'final_score': [81, 90, 58, 45, 95, 60, 81]
}
df = pd.DataFrame(data)

# 2. Drop duplicates
df = df.drop_duplicates()

# 3. Clean Text Categories
df['city'] = df['city'].astype('string').str.strip().str.lower()
df['city'] = df['city'].replace({'islamabad': 'islamabad', 'lahore': 'lahore', 'karachi': 'karachi'})

# 4. Handle Out-of-Bound / Invalid Values
df.loc[(df['attendance'] < 0) | (df['attendance'] > 100), 'attendance'] = np.nan

# 5. Outlier Detection (IQR on study_hours)
Q1 = df['study_hours'].quantile(0.25)
Q3 = df['study_hours'].quantile(0.75)
IQR = Q3 - Q1
df.loc[(df['study_hours'] < (Q1 - 1.5 * IQR)) | (df['study_hours'] > (Q3 + 1.5 * IQR)), 'study_hours'] = np.nan

# 6. Split Features and Target
X = df.drop(columns=['student_id', 'final_score'])
y = df['final_score']

# 7. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 8. Preprocessing Pipelines
num_cols = ['study_hours', 'attendance', 'previous_score']
cat_cols = ['city', 'internet_access']

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print("--- PREPROCESSING COMPLETE ---")
print("Processed Training Features Shape:", X_train_proc.shape)
print("Processed Testing Features Shape:", X_test_proc.shape)
print("\nTransformed Training Array (First 2 Rows):\n", X_train_proc[:2])

--- PREPROCESSING COMPLETE ---
Processed Training Features Shape: (4, 8)
Processed Testing Features Shape: (2, 8)

Transformed Training Array (First 2 Rows):
 [[ 1.41421356  0.         -0.38605074  0.          1.          0.
   1.          0.        ]
 [ 0.          0.         -0.15208059  0.          1.          0.
   1.          0.        ]]


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Load data
df = pd.read_csv("student_performance.csv")

# Feature/Target Separation
X = df.drop(columns=["final_score"])
y = df["final_score"]

numeric_features = ["study_hours", "attendance", "previous_score"]
categorical_features = ["city", "internet_access"]

# Split data (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Numeric & Categorical Pipelines
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine Transformer
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Fit on Train, Transform both Train & Test independently
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

print("Train shape:", X_train_ready.shape)
print("Test shape:", X_test_ready.shape)

Train shape: (4, 8)
Test shape: (1, 8)
